## CLIP Image-Text Search

This project demonstrates how to build a **prompt-to-image retrieval system** using a pre-built image-caption dataset inspired by ImageNet and Towhee. Instead of training from scratch, we leverage the powerful [Sentence-Transformers](https://www.sbert.net/) CLIP model (`clip-ViT-B-32`) to embed both **textual prompts** and **image features**, and retrieve the most relevant images based on **cosine similarity**.

Running this project on **[Saturn Cloud](https://saturncloud.io/)** gives you:

* 🚀 GPU-accelerated inference for large embedding models
* 🧩 Reproducibility across experiments and environments
* 👥 Collaborative Jupyter notebooks with persistent storage
* 💼 Enterprise-ready tools with easy deployment to production

This is an ideal template for reverse image search, image tagging, and multimodal AI applications.

Installing dependency for the project.

In [8]:
!pip install -q \
  sentence-transformers \
  scikit-learn \
  gradio \
  pandas \
  pillow \
  tqdm


This block downloads a lightweight **reverse image search dataset** from Towhee’s GitHub releases. It includes training and test image directories and a CSV mapping file that pairs image paths with labels, which we'll treat as "pseudo-captions".

In [ ]:
# 🔽 Download and unzip the Towhee reverse image search dataset
!wget https://github.com/towhee-io/examples/releases/download/data/reverse_image_search.zip
!unzip -q reverse_image_search.zip -d images_folder

This block downloads a lightweight **reverse image search dataset** from Towhee’s GitHub releases. It includes training and test image directories and a CSV mapping file that pairs image paths with labels, which we'll treat as "pseudo-captions".

we load the `reverse_image_search.csv` metadata file to extract:

* ✅ Full image paths
* ✅ Corresponding class labels (used as text prompts)

These are the foundation for generating our dual embeddings.

In [ ]:
import pandas as pd
from pathlib import Path

# Paths
csv_path = Path("images_folder/reverse_image_search.csv")
root_dir = Path("images_folder")  # Base directory for relative paths in CSV

# Load metadata
df = pd.read_csv(csv_path)

# Construct full image paths and labels
image_paths = [root_dir / Path(p) for p in df["path"]]
captions = df["label"].tolist()  # Use labels as pseudo-captions
print(f"✅ Loaded {len(image_paths)} images with labels")


We load **`clip-ViT-B-32`**, a dual encoder from `sentence-transformers` capable of embedding both text and images in the same latent space. This enables natural language to be matched directly with visual content.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

# Load multi-modal embedding model
model = SentenceTransformer("clip-ViT-B-32").to(device)

This section encodes:

* **Image features** via `model.encode(image)`
* **Text prompts (labels)** via `model.encode(label)`

Both embeddings are saved as numpy arrays for efficient similarity comparisons.

In [ ]:
from PIL import Image
import numpy as np
from tqdm import tqdm

text_embeddings = []
image_embeddings = []
valid_paths = []
valid_labels = []

print("🔍 Generating embeddings...")
for path, label in tqdm(zip(image_paths, captions), total=len(image_paths)):
    try:
        img = Image.open(path).convert("RGB")

        # Generate image embedding
        with torch.no_grad():
            img_emb = model.encode(img, convert_to_tensor=True).cpu().numpy()

        # Generate text embedding
        with torch.no_grad():
            txt_emb = model.encode(label, convert_to_tensor=True).cpu().numpy()

        image_embeddings.append(img_emb)
        text_embeddings.append(txt_emb)
        valid_paths.append(path)
        valid_labels.append(label)

    except Exception as e:
        print(f"❌ Skipped {path.name}: {e}")
        continue

print(f"✅ Embedded {len(valid_paths)} images")

All preprocessed data is saved for later use:

* 🔐 `image_embeddings.npy`
* 🔐 `text_embeddings.npy`
* 🔐 `metadata.json` (image paths + labels)

In [ ]:
import json
import numpy as np

save_dir = Path("data/processed")
save_dir.mkdir(parents=True, exist_ok=True)

# Save embeddings
np.save(save_dir / "image_embeddings.npy", np.vstack(image_embeddings))
np.save(save_dir / "text_embeddings.npy", np.vstack(text_embeddings))

# Save paths and labels as metadata
metadata = {
    "paths": [str(p) for p in valid_paths],
    "labels": valid_labels
}
with open(save_dir / "metadata.json", "w") as f:
    json.dump(metadata, f)

print("💾 Embeddings and metadata saved.")

Using **cosine similarity**, this function:

* Embeds the input query
* Compares it to all text embeddings
* Returns the top-k matches based on similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def search_images(prompt, k=3):
    # Load data
    image_embeddings = np.load("data/processed/image_embeddings.npy")
    text_embeddings = np.load("data/processed/text_embeddings.npy")

    with open("data/processed/metadata.json", "r") as f:
        metadata = json.load(f)

    paths = metadata["paths"]
    labels = metadata["labels"]

    # Encode query
    query_emb = model.encode(prompt, convert_to_tensor=True).cpu().numpy().reshape(1, -1)

    # Compare to text embeddings
    similarities = cosine_similarity(query_emb, text_embeddings)[0]
    top_indices = similarities.argsort()[::-1][:k]

    results = [{"path": paths[i], "label": labels[i], "score": float(similarities[i])} for i in top_indices]
    return results


This section builds an interactive **Gradio app** so users can enter a natural language prompt and view the top matching images.

Perfect for demos, prototyping, or deploying as a user-facing search tool.

In [ ]:
import gradio as gr
from PIL import Image

def search_and_display(query, k):
    results = search_images(query, k)
    return [(r["path"], f"{r['label']} ({r['score']:.2f})") for r in results]

gr.Interface(
    fn=search_and_display,
    inputs=[
        gr.Textbox(label="Enter a prompt (e.g. 'brain coral', 'a hound dog')"),
        gr.Slider(1, 5, value=3, step=1, label="Number of results")
    ],
    outputs=gr.Gallery(label="Matching Images"),
    title="🔍 Prompt → Image Search",
    description="Type a label or prompt to search for similar images in the ImageNet-style dataset"
).launch(share=True)


## ✅ Conclusion (Tailored for Saturn Cloud)

In this project, you built a **prompt-to-image retrieval system** using a CLIP-based model from `sentence-transformers`, applied to a lightweight ImageNet-style dataset.

By running this on **[Saturn Cloud](https://saturncloud.io/)**, you benefit from:

* 🚀 Fast, GPU-accelerated model inference for embeddings
* 💾 Persistent storage across Jupyter sessions for data, models, and results
* 🔁 Easy reproducibility of workflows in collaborative teams
* 📦 Seamless deployment to production-ready environments

Whether you're prototyping AI search tools or deploying image classification pipelines, Saturn Cloud’s infrastructure makes it easy to scale from notebooks to production.

---

### 🧭 Want to Learn More?

* 📚 [Saturn Cloud Documentation](https://saturncloud.io/docs/)
* 🔨 [Saturn Cloud Templates](https://saturncloud.io/resources/templates/)
* 🌐 [Saturn Cloud Home](https://saturncloud.io/)